In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchtext.datasets import Multi30k
from torchtext.data import Field, BucketIterator, Example, Dataset
import numpy as np
import spacy
import random
import torchtext.datasets as datasets
import os
import urllib.request
import gzip
import shutil
import torchinfo

/home/mayank/ai/lib/python3.12/site-packages/torch/cuda/__init__.py:141: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [3]:
torch.cuda.is_available()

False

In [4]:
DATA_URLS = {
    "train.de": "https://github.com/multi30k/dataset/raw/master/data/task1/raw/train.de.gz",
    "train.en": "https://github.com/multi30k/dataset/raw/master/data/task1/raw/train.en.gz",
    "val.de": "https://github.com/multi30k/dataset/raw/master/data/task1/raw/val.de.gz",
    "val.en": "https://github.com/multi30k/dataset/raw/master/data/task1/raw/val.en.gz",
    "test.de": "https://github.com/multi30k/dataset/raw/master/data/task1/raw/test_2016_flickr.de.gz",
    "test.en": "https://github.com/multi30k/dataset/raw/master/data/task1/raw/test_2016_flickr.en.gz",
}

DATA_DIR = "multi30k"

# Step 1: Download and Extract Data
def download_and_extract():
    os.makedirs(DATA_DIR, exist_ok=True)
    for filename, url in DATA_URLS.items():
        gz_file = os.path.join(DATA_DIR, filename + ".gz")
        txt_file = os.path.join(DATA_DIR, filename)

        # Download file if it doesn't exist
        if not os.path.exists(txt_file):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, gz_file)

            # Extract file
            with gzip.open(gz_file, "rb") as f_in, open(txt_file, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
            os.remove(gz_file)
            print(f"Extracted {filename}")

# Step 2: Load Data into PyTorch Dataset
def load_dataset(src_file, trg_file, src_field, trg_field):
    examples = []
    with open(src_file, "r", encoding="utf-8") as src_f, open(trg_file, "r", encoding="utf-8") as trg_f:
        for src, trg in zip(src_f, trg_f):
            examples.append(Example.fromlist([src.strip(), trg.strip()], fields=[("src", src_field), ("trg", trg_field)]))
    return Dataset(examples, fields={"src": src_field, "trg": trg_field})

# Step 3: Define Tokenizers
german = Field(tokenize="spacy", tokenizer_language="de_core_news_sm", lower=True, init_token = "<sos>", eos_token = "<eos>")
english = Field(tokenize="spacy", tokenizer_language="en_core_web_sm", lower=True, init_token = "<sos>", eos_token = "<eos>")

# Download, extract, and load dataset
download_and_extract()

train_data = load_dataset(os.path.join(DATA_DIR, "train.de"), os.path.join(DATA_DIR, "train.en"), german, english)
val_data = load_dataset(os.path.join(DATA_DIR, "val.de"), os.path.join(DATA_DIR, "val.en"), german, english)
test_data = load_dataset(os.path.join(DATA_DIR, "test.de"), os.path.join(DATA_DIR, "test.en"), german, english)

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# Step 4: Create DataLoader for Training
BATCH_SIZE = 32
train_iterator, val_iterator, test_iterator = BucketIterator.splits(
    (train_data, val_data, test_data),
    batch_size=BATCH_SIZE,
    device="cuda" if torch.cuda.is_available() else "cpu",
    sort_within_batch=True,
    sort_key=lambda x: len(x.src)
)

print("Data loading complete! ✅")


Train samples: 29000
Validation samples: 1014
Test samples: 1000
Data loading complete! ✅


In [5]:
german.build_vocab(train_data, max_size = 10000, min_freq = 1)
english.build_vocab(train_data, max_size = 10000, min_freq = 1)

In [6]:
class Encoder(nn.Module):
    def __init__(self, input_size, embedding_size, hidden_size, num_layers, dropout):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.drop = nn.Dropout(dropout)
        self.input_size = input_size
        self.embedding = nn.Embedding(input_size, embedding_size)
        self.lstm = nn.LSTM(embedding_size, hidden_size, num_layers, dropout = dropout, bidirectional = True)
        self.fc = nn.Linear(hidden_size*2, hidden_size)

    def forward(self, x):
        
        ## x shape -> (seq_length, N) where N is batch size
        embedded = self.embedding(x)
        embedded = self.drop(embedded)
        ##x shape -> (seq_legth, N, embedding_size)
        
        encoder_states, (hidden, cell) = self.lstm(embedded)
        ## encoder_states shape -> (seq_length, N, hidden_size)
        ## hidden shape -> (num_layers*directions(2), N, hidden_size)

        hidden = self.fc(torch.cat((hidden[0:1], hidden[1:2]), dim = 2))
        cell = self.fc(torch.cat((cell[0:1], cell[1:2]), dim = 2))
        ## hidden[0:1] gets the hidden states for one direction and hidden[1:2] gets the hidden states for other direction. Same for cell

        return (encoder_states, hidden, cell)

In [7]:
class Decoder(nn.Module):
    def __init__(self, input_size, embedding_size, hidden_size, output_size, num_layers, dropout):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.drop = nn.Dropout(dropout)
        self.embedding = nn.Embedding(input_size, embedding_size)

        self.lstm = nn.LSTM(hidden_size*2 + embedding_size, hidden_size, num_layers, dropout = dropout)

        self.energy = nn.Linear(hidden_size*3, 1)
        
        self.softmax = nn.Softmax(dim = 0)

        self.relu = nn.ReLU()

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, encoder_states, hidden, cell):
        ## shape of x -> (N) but we want it to be (1, N) because our decode is gonna predict one word at a time

        x = x.unsqueeze(0)
        embedded = self.drop(self.embedding(x))
        ## x shape -> (1, N, embedding_size)

        seq_length = encoder_states.shape[0]
        hidden_reshaped = hidden.repeat(seq_length, 1, 1) ## this basically reapeats the tensor hidden 'seq_length' times on the x axis

        energy = self.relu(self.energy(torch.cat((hidden_reshaped, encoder_states), dim = 2)))
        ## energy shape -> (seq_length, N, 1)

        attention = self.softmax(energy)
        ## attention shape -> (seq_length, N, 1)

        attention = attention.permute(1, 2, 0)
        ## attention shape -> (N, 1, seq_length)

        encoder_states = encoder_states.permute(1, 0, 2)
        ## ecoder_states shape -> (N, seq_legth, hidden_size*2)

        context_vector = torch.bmm(attention, encoder_states).permute(1, 0, 2) ## matrix multiplication of encoder_states with their attention weights and changing the shape
        ## context_vector shape -> (1, N, hidden_size*2)

        lstm_input = torch.cat((context_vector, embedded), dim = 2)
        ## lstm_input shape -> (1, N, hidden_size*2 + embedding_size)

        outputs, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        ## outputs shape -> (1, N, hidden_size)

        predictions = self.fc(outputs.squeeze(0))
        ## predictions shape -> (N, output_size)

        return (predictions, hidden, cell)

In [8]:
class Attention(nn.Module):
    def __init__(self, encoder, decoder):
        super(Attention, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, target, teacher_force_ratio = 0.5): ##teacher_force_ratio is the probability of using teacher forcing that is the actual target value that should be passed in the input of the decoder at any lstm cell. By making it 50% we are making sure that 50% of the time the actual target value is passed and 50% of the time the predicted value is passed. We cannot make it 100% because then only correct value will be passed and it will not be able to predict the target value on its own.
        batch_size = target.shape[1]
        target_len = target.shape[0]
        target_vocab_size = len(german.vocab)

        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(device)

        (encoder_states, hidden, cell) = self.encoder(source)

        ## grab start token
        x = target[0, :]

        for t in range(1, target_len):
            (output, hidden, cell) = self.decoder(x, encoder_states, hidden, cell)

            outputs[t] = output
            ## output shape -> (N, german_vocab_size)
            best_guess = output.argmax(1)

            x = target[t] if random.random() < teacher_force_ratio else best_guess

        return outputs

In [9]:
num_epochs = 100
learning_rate = 0.001
batch_size = 64

In [10]:
load_model = True
input_size_encoder = len(english.vocab)
input_size_decoder = len(german.vocab)
output_size = len(german.vocab)
encoder_embedding_size = 256
decoder_embedding_size = 256
hidden_size = 512
num_layers = 1
encoder_dropout = 0.5
decoder_dropout = 0.5

In [11]:
train_iterator, valid_iterator, test_iterator = BucketIterator.splits((train_data, val_data, test_data), batch_size = batch_size, sort_within_batch = True, sort_key = lambda x: len(x.src), device = device)

In [12]:
encoder_net = Encoder(input_size_encoder, encoder_embedding_size, hidden_size, num_layers, encoder_dropout).to(device)
decoder_net = Decoder(input_size_decoder, decoder_embedding_size, hidden_size, output_size, num_layers, decoder_dropout).to(device)
model = Attention(encoder_net, decoder_net).to(device)

/home/mayank/ai/lib/python3.12/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [13]:
pad_idx = german.vocab.stoi[german.pad_token]
criterion = nn.CrossEntropyLoss(ignore_index = pad_idx)
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [16]:
if load_model:
    model.load_state_dict(torch.load("AttentionModel.pt", map_location = torch.device('cpu')))

In [17]:
print(f"Total parameters to train: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Total parameters to train: 17555477


In [18]:
def predict_sentence(model, device, sentence):
    def tokenize_sentence(sentence):
        tokens = [word for word in sentence.split()]
        indices = [english.vocab.stoi[word] for word in tokens]
        tensor = torch.tensor(indices).unsqueeze(1).to(device)
        return tensor
    
    inp_data = tokenize_sentence(sentence)

    encoder_states, hidden, cell = model.encoder(inp_data)
    
    output_sentence = []
    x = torch.tensor([[german.vocab.stoi["<sos>"]]]).to(device)
    x = x[0, :]

    while True:
        (output, hidden, cell) = model.decoder(x, encoder_states, hidden, cell)
        
        pred_token_idx = output.argmax(1)
        
        pred_token = german.vocab.itos[pred_token_idx.item()]
    
        if pred_token == "<eos>": break
        else: output_sentence.append(pred_token)
        
        # Update input for next iteration
        x = torch.tensor([[pred_token_idx]]).to(device)
        x = x[0, :]

    # Print predicted output sentence
    pred_output = " ".join(output_sentence)
    return pred_output

In [18]:
for epoch in range(num_epochs):
    print(f"Training epoch {epoch+1}/{num_epochs}")
    losses = []

    for batch_idx, batch in enumerate(train_iterator):
        inp_data = batch.src.to(device)
        target = batch.trg.to(device)
        inp_data = torch.clamp(inp_data, max = input_size_encoder - 1)

        '''print(inp_data.shape, target.shape)
        print(torch.max(inp_data))'''
        output = model(inp_data, target)
        ## output shape -> (trg_len, batch_size, output_dim)

        output = output[1:].reshape(-1, output.shape[2])
        target = target[1:].reshape(-1)

        optimizer.zero_grad()
        loss = criterion(output, target)
        losses.append(loss.item())
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1) ## to prevent exploding gradient problem

        optimizer.step()
    torch.save(model.state_dict(), "AttentionModel.pt")
    print(f"Loss: {np.mean(np.array(losses))}")
    

Training epoch 1/100
Loss: 4.6041686666169355
Training epoch 2/100
Loss: 3.721776002829295
Training epoch 3/100
Loss: 3.3659575405624995
Training epoch 4/100
Loss: 3.111388504767733
Training epoch 5/100
Loss: 2.9070469471851634
Training epoch 6/100
Loss: 2.7345523658302913
Training epoch 7/100
Loss: 2.5984508959732393
Training epoch 8/100
Loss: 2.4711942659600714
Training epoch 9/100
Loss: 2.3617841018454095
Training epoch 10/100
Loss: 2.2519682887367214
Training epoch 11/100
Loss: 2.1760405260035647
Training epoch 12/100
Loss: 2.1020767226618293
Training epoch 13/100
Loss: 2.018643076724418
Training epoch 14/100
Loss: 1.9500213937612356
Training epoch 15/100
Loss: 1.8919151027559709
Training epoch 16/100
Loss: 1.8243617844739148
Training epoch 17/100
Loss: 1.7517249194273339
Training epoch 18/100
Loss: 1.7247503067690895
Training epoch 19/100
Loss: 1.6743395209049863
Training epoch 20/100
Loss: 1.632978704269762
Training epoch 21/100
Loss: 1.6055954853606118
Training epoch 22/100
Loss